# Module 4: Persistence & Memory Checkpointing

In this notebook, we will explore:
1. **MemorySaver**: Compiling a graph with an active state checkpointer saver.
2. **Thread Isolation**: Isolating conversational threads based on configurations.
3. **State History**: Inspecting past snapshots and checkpoint coordinates.
4. **Time Travel**: Modifying past states and resuming execution from historical forks.

### Step 1: Initialize Chat Model Connection

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(dotenv_path="../../../langchain/.env")

model = ChatOpenAI(
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    model_name="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0.3,
)
print("Model client connected!")

---
## 1. Setting up a Conversational Graph with Memory

We will build a simple chatbot graph using `MessagesState`. When compiling, we attach a `MemorySaver` to checkpoint every state transition.

In [ ]:
from langgraph.graph import START, END, MessagesState, StateGraph
from langgraph.checkpoint.memory import MemorySaver

def chatbot_node(state: MessagesState) -> dict:
    print("--- Executing Chatbot Node ---")
    messages = state["messages"]
    
    # Invoke LLM
    response = model.invoke(messages)
    
    # Return message update to append
    return {"messages": [response]}

# Build Graph
builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot_node)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

# 1. Initialize checkpointer
memory_checkpointer = MemorySaver()

# 2. Compile graph passing checkpointer
graph = builder.compile(checkpointer=memory_checkpointer)
print("Conversational Graph Compiled with MemorySaver!")

---
## 2. Verifying Session Isolation (Thread IDs)

Let's run a conversation on `thread_1` declaring our name, then query the bot. We will verify that a separate `thread_2` query remains completely isolated.

In [ ]:
from langchain_core.messages import HumanMessage

# Configuration for Thread 1
config_1 = {"configurable": {"thread_id": "thread_bob"}}

# Turn 1: Declare Name
res1 = graph.invoke(
    {"messages": [HumanMessage(content="Hi, my name is Bob. I enjoy biking.")]},
    config=config_1
)
print("Bob Turn 1 Response:\n", res1["messages"][-1].content)

# Turn 2: Query Back
res2 = graph.invoke(
    {"messages": [HumanMessage(content="What was my name again?")]},
    config=config_1
)
print("\nBob Turn 2 Response:\n", res2["messages"][-1].content)

Now let's query the graph using `thread_charlie`. It should have no record of the previous Bob conversation.

In [ ]:
# Configuration for Thread 2
config_2 = {"configurable": {"thread_id": "thread_charlie"}}

res_charlie = graph.invoke(
    {"messages": [HumanMessage(content="What was my name again?")]},
    config=config_2
)
print("Charlie Response:\n", res_charlie["messages"][-1].content)

---
## 3. Inspecting State History

Let's look at the database logs. We will print the list of checkpoints saved for `thread_bob`.

In [ ]:
# Fetch state history records
history_records = list(graph.get_state_history(config_1))

print(f"Total saved checkpoints for Bob's thread: {len(history_records)}\n")

for snapshot in history_records:
    print(f"Checkpoint Config Coordinates:")
    print(f"  {snapshot.config}")
    print(f"Next node to run: {snapshot.next}")
    print(f"State Messages Count: {len(snapshot.values.get('messages', []))}")
    print(f"Last Message in Snapshot: {repr(snapshot.values.get('messages')[-1].content[:60])}...")
    print("-" * 70)

---
## 4. Time Travel (Updating State & Forking)

Let's travel back in time. We will target the state snapshot right after Bob said "Hi, my name is Bob. I enjoy biking.", update the state to say the name is "David" who enjoys "kayaking", and resume execution.

In [ ]:
# Let's identify the checkpoint right after Bob's first message.
# That corresponds to the second snapshot from the bottom of our history list.
target_snapshot = history_records[-1] # The initial setup state before Turn 2 started
print("Targeting Checkpoint:", target_snapshot.config)

# Fetch the messages at that snapshot
current_msgs = target_snapshot.values["messages"]
print("\nMessages at checkpoint:")
for msg in current_msgs:
    print(f"  [{type(msg).__name__}]: {msg.content}")

Now we overwrite/update the state at that specific checkpoint by passing an updated message with the same ID, or updating the state list. Let's update the human message.

In [ ]:
from langchain_core.messages import AIMessage

# Let's change Bob's first message ID content to David
bob_first_msg_id = current_msgs[0].id
updated_human_msg = HumanMessage(
    content="Hi, my name is David. I enjoy kayaking.",
    id=bob_first_msg_id
)

print("Applying State Update to historical checkpoint...")
fork_config = graph.update_state(
    target_snapshot.config,
    {"messages": [updated_human_msg]},
    as_node="chatbot" # Identify which node is making the change
)

print("New fork config coordinates:")
print(fork_config)

Now let's query the model *resuming* from this newly updated fork configuration. We pass `None` as the input to tell the graph to resume execution from its current state values.

In [ ]:
print("Resuming graph execution from fork...")
# Resume chatbot execution loop using fork configuration coordinates
graph.invoke(None, config=fork_config)

# Now query the bot about our name on this thread
res_fork_query = graph.invoke(
    {"messages": [HumanMessage(content="What is my name and what do I enjoy?")]},
    config=fork_config
)

print("\nAI Reply from Forked Timeline:")
print(res_fork_query["messages"][-1].content)

Excellent! The bot replied with "David" and "kayaking". The state was successfully modified in the past and forked into a new timeline.